[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abhisheksreesaila/mojo-gpu-tutorials/blob/main/004_thread_mgmt.ipynb)

In [ ]:
%%capture
!pip install mojo

## 🎯 Kernel Tutorial: Adding 10 with Boundary Conditions 🚀

### 🎬 Video Overview
Learn how to implement a CUDA kernel that adds 10 to each element of a vector/matrix, but with a twist - handling cases where you have fewer data elements than threads! 🔥

### 🧠 The Core Concept
📝 What We're Building
A kernel that:

- ✅ Takes input vector/matrix a
- ✅ Adds 10 to each element
- ✅ Stores result in output
- ⚠️ But: Handles boundary conditions when threads > data elements

## Until now

<img src="../../assets/004_thread_mgmt-1.png" width="600" height="600">

## Today

<img src="../../assets/004_thread_mgmt-2.png" width="800" height="600">

In [1]:
import mojo.notebook

In [4]:
%%mojo

from memory import UnsafePointer
from gpu import thread_idx
from gpu.host import DeviceContext


comptime SIZE = 4
comptime BLOCKS_PER_GRID = 1
comptime THREADS_PER_BLOCK = (8, 1)
comptime dtype = DType.float32


fn add_10(
    output: UnsafePointer[Scalar[dtype], MutAnyOrigin],
    a: UnsafePointer[Scalar[dtype], MutAnyOrigin],
    size: UInt,
):
    var i = thread_idx.x

    # 🛡️ CRITICAL BOUNDARY CHECK!
    if i < size:  # 🚫 Threads with idx >= n do nothing and return safely
        output[i] = a[i] + 10.0

def main():

    # BOILER PLATE
    var ctx = DeviceContext()
    var out = ctx.enqueue_create_buffer[dtype](SIZE)
    out.enqueue_fill(0)
    var a = ctx.enqueue_create_buffer[dtype](SIZE)
    a.enqueue_fill(0)
    with a.map_to_host() as a_host:
        for i in range(SIZE):
            a_host[i] = i

    ctx.enqueue_function_checked[add_10, add_10](
            out,
            a,
            UInt(SIZE),
            grid_dim=BLOCKS_PER_GRID,
            block_dim=THREADS_PER_BLOCK,
        )
    
    ctx.synchronize()
    
    with out.map_to_host() as out_host:
        print(out_host)

HostBuffer([10.0, 11.0, 12.0, 13.0])



### 2nd Kernel

<img src="../../assets/004_thread_mgmt-3.png" width="600" height="400">


### 📐 Visual Layout of BLOCKS, THREADS within a GRID

```
Thread  |    0  1  2  3
-------------------------
block 0 | [  0  1  2  3 ]   ← IDs: 0, 1, 2, 3
block 1 | [  0  1  2  3 ]   ← IDs: 4, 5, 6, 7
block 2 | [  0  1  2  3 ]   ← IDs: 8, 9, 10, 11  
block 3 | [  0  1  2  3 ]   ← IDs: 12, 13, 14, 15
```

In [6]:
%%mojo

from memory import UnsafePointer
from gpu import thread_idx, block_idx, block_dim
from gpu.host import DeviceContext

comptime SIZE = 9
comptime BLOCKS_PER_GRID = (3, 1)
comptime THREADS_PER_BLOCK =  (4, 1)
comptime dtype = DType.float32

fn add_10(
    output: UnsafePointer[Scalar[dtype], MutAnyOrigin],
    a: UnsafePointer[Scalar[dtype], MutAnyOrigin],
    size: UInt,
):

    var i = block_dim.x * block_idx.x + thread_idx.x #<<< this is the only change. GEt the index correctly considering multiple blocks
    # 🛡️ CRITICAL BOUNDARY CHECK!
    if i < size:  # 🚫 Threads with idx >= n do nothing and return safely
        output[i] = a[i] + 10.0

def main():

    # BOILER PLATE
    var ctx = DeviceContext()
    var out = ctx.enqueue_create_buffer[dtype](SIZE)
    out.enqueue_fill(0)
    var a = ctx.enqueue_create_buffer[dtype](SIZE)
    a.enqueue_fill(0)
    with a.map_to_host() as a_host:
        for i in range(SIZE):
            a_host[i] = i

    ctx.enqueue_function_checked[add_10, add_10](
            out,
            a,
            UInt(SIZE),
            grid_dim=BLOCKS_PER_GRID,
            block_dim=THREADS_PER_BLOCK,
        )
    
    ctx.synchronize()
    
    with out.map_to_host() as out_host:
        print(out_host)

HostBuffer([10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0])



## 3rd Kernel

Apply the 2 priniciples

- convert the local ID -> global ID
- Use the "i<size" but in 2 dimensions

<img src="../../assets/004_thread_mgmt-4.png" width="600" height="400">

```

┌─────────────────────────────────────┬─────────────────────────────────────┐
│         Block (0,0)                 │         Block (1,0)                 │
│  ┌──────────┬──────────┬──────────┐ │  ┌──────────┬──────────┬──────────┐ │
│  │ B(0,0)   │ B(0,0)   │ B(0,0)   │ │  │ B(1,0)   │ B(1,0)   │ B(1,0)   │ │
│  │ T(0,0)   │ T(1,0)   │ T(2,0)   │ │  │ T(0,0)   │ T(1,0)   │ T(2,0)   │ │
│  │ ───────  │ ───────  │ ───────  │ │  │ ───────  │ ───────  │ ───────  │ │
│  │ a[0,0]   │ a[0,1]   │ a[0,2]   │ │  │ a[0,3]   │ a[0,4]   │ OUT❌    │ │
│  ├──────────┼──────────┼──────────┤ │  ├──────────┼──────────┼──────────┤ │
│  │ B(0,0)   │ B(0,0)   │ B(0,0)   │ │  │ B(1,0)   │ B(1,0)   │ B(1,0)   │ │
│  │ T(0,1)   │ T(1,1)   │ T(2,1)   │ │  │ T(0,1)   │ T(1,1)   │ T(2,1)   │ │
│  │ ───────  │ ───────  │ ───────  │ │  │ ───────  │ ───────  │ ───────  │ │
│  │ a[1,0]   │ a[1,1]   │ a[1,2]   │ │  │ a[1,3]   │ a[1,4]   │ OUT❌    │ │
│  ├──────────┼──────────┼──────────┤ │  ├──────────┼──────────┼──────────┤ │
│  │ B(0,0)   │ B(0,0)   │ B(0,0)   │ │  │ B(1,0)   │ B(1,0)   │ B(1,0)   │ │
│  │ T(0,2)   │ T(1,2)   │ T(2,2)   │ │  │ T(0,2)   │ T(1,2)   │ T(2,2)   │ │
│  │ ───────  │ ───────  │ ───────  │ │  │ ───────  │ ───────  │ ───────  │ │
│  │ a[2,0]   │ a[2,1]   │ a[2,2]   │ │  │ a[2,3]   │ a[2,4]   │ OUT❌    │ │
│  └──────────┴──────────┴──────────┘ │  └──────────┴──────────┴──────────┘ │
├─────────────────────────────────────┼─────────────────────────────────────┤
│         Block (0,1)                 │         Block (1,1)                 │
│  ┌──────────┬──────────┬──────────┐ │  ┌──────────┬──────────┬──────────┐ │
│  │ B(0,1)   │ B(0,1)   │ B(0,1)   │ │  │ B(1,1)   │ B(1,1)   │ B(1,1)   │ │
│  │ T(0,0)   │ T(1,0)   │ T(2,0)   │ │  │ T(0,0)   │ T(1,0)   │ T(2,0)   │ │
│  │ ───────  │ ───────  │ ───────  │ │  │ ───────  │ ───────  │ ───────  │ │
│  │ a[3,0]   │ a[3,1]   │ a[3,2]   │ │  │ a[3,3]   │ a[3,4]   │ OUT❌    │ │
│  ├──────────┼──────────┼──────────┤ │  ├──────────┼──────────┼──────────┤ │
│  │ B(0,1)   │ B(0,1)   │ B(0,1)   │ │  │ B(1,1)   │ B(1,1)   │ B(1,1)   │ │
│  │ T(0,1)   │ T(1,1)   │ T(2,1)   │ │  │ T(0,1)   │ T(1,1)   │ T(2,1)   │ │
│  │ ───────  │ ───────  │ ───────  │ │  │ ───────  │ ───────  │ ───────  │ │
│  │ a[4,0]   │ a[4,1]   │ a[4,2]   │ │  │ a[4,3]   │ a[4,4]   │ OUT❌    │ │
│  ├──────────┼──────────┼──────────┤ │  ├──────────┼──────────┼──────────┤ │
│  │ B(0,1)   │ B(0,1)   │ B(0,1)   │ │  │ B(1,1)   │ B(1,1)   │ B(1,1)   │ │
│  │ T(0,2)   │ T(1,2)   │ T(2,2)   │ │  │ T(0,2)   │ T(1,2)   │ T(2,2)   │ │
│  │ ───────  │ ───────  │ ───────  │ │  │ ───────  │ ───────  │ ───────  │ │
│  │ OUT❌    │ OUT❌    │ OUT❌    │ │  │ OUT❌    │ OUT❌    │ OUT❌    │ │
│  └──────────┴──────────┴──────────┘ │  └──────────┴──────────┴──────────┘ │
└─────────────────────────────────────┴─────────────────────────────────────┘

Legend:
  B(x,y) = Block Index (block_idx.x, block_idx.y)
  T(x,y) = Thread Index (thread_idx.x, thread_idx.y)
  a[r,c] = Matrix Element at [row, col]
  OUT❌  = Out of bounds for 5×5 matrix


```

In [ ]:

%%mojo

from gpu import thread_idx, block_idx, block_dim
from memory import UnsafePointer
from gpu import thread_idx
from gpu.host import DeviceContext


alias SIZE_3k = 5
alias BLOCKS_PER_GRID_3k = (2, 2)
alias THREADS_PER_BLOCK_3k = (3, 3)
alias dtype_3k = DType.float32


fn add_10_3k(
    output: UnsafePointer[Scalar[dtype]],
    a: UnsafePointer[Scalar[dtype]],
    size: Int,
):
    row = block_dim.y * block_idx.y + thread_idx.y
    col = block_dim.x * block_idx.x + thread_idx.x
    
    if row < size and col < size:
        output[row * size + col] = a[row * size + col] + 10.0

# output[0 * 5 + 0] = output[0] = a[0] + 10
# output[0 * 5 + 1] = output[1] = a[1] + 10
# output[0 * 5 + 2] = output[2] = a[2] + 10
# output[0 * 5 + 3] = output[3] = a[3] + 10
# output[0 * 5 + 4] = output[4] = a[4] + 10

# second row
## output[1 * 5 + 0] = output[5] = a[5] + 10
## output[1 * 5 + 1] = output[6] = a[6] + 10
...

print("Adding Kernel Here")


#BOILER PLATE 
var ctx_3k = DeviceContext()
out_3k = ctx_3k.enqueue_create_buffer[dtype_3k](SIZE_3k * SIZE_3k)
out_3k.enqueue_fill(0)
a_3k = ctx_3k.enqueue_create_buffer[dtype_3k](SIZE_3k * SIZE_3k)
a_3k.enqueue_fill(0)
with a_3k.map_to_host() as a_3k_host:
    for j in range(SIZE_3k):
        for i in range(SIZE_3k):
            k = j * SIZE_3k + i
            a_3k_host[k] = k

print(a_3k)

ctx_3k.enqueue_function[add_10_3k](
            out_3k,
            a_3k,
            SIZE_3k,
            grid_dim=BLOCKS_PER_GRID_3k,
            block_dim=THREADS_PER_BLOCK_3k,
        )

ctx_3k.synchronize()

with out_3k.map_to_host() as out3k_host:
    for i in range(SIZE_3k):
        for j in range(SIZE_3k):
            print(out3k_host[i * SIZE_3k + j])


In [10]:
%%mojo

from memory import UnsafePointer
from gpu import thread_idx, block_idx, block_dim
from gpu.host import DeviceContext

comptime SIZE = 5
comptime BLOCKS_PER_GRID = (2, 2)
comptime THREADS_PER_BLOCK =  (3, 3)
comptime dtype = DType.float32

fn add_10(
    output: UnsafePointer[Scalar[dtype],MutAnyOrigin],
    a: UnsafePointer[Scalar[dtype],MutAnyOrigin],
    size: UInt,
):
    row = block_dim.y * block_idx.y + thread_idx.y
    col = block_dim.x * block_idx.x + thread_idx.x
    
    if row < size and col < size:
        output[row * size + col] = a[row * size + col] + 10.0

# output[0 * 5 + 0] = output[0] = a[0] + 10
# output[0 * 5 + 1] = output[1] = a[1] + 10
# output[0 * 5 + 2] = output[2] = a[2] + 10
# output[0 * 5 + 3] = output[3] = a[3] + 10
# output[0 * 5 + 4] = output[4] = a[4] + 10

# second row
## output[1 * 5 + 0] = output[5] = a[5] + 10
## output[1 * 5 + 1] = output[6] = a[6] + 10

def main():

    # BOILER PLATE
    var ctx = DeviceContext()
    var out = ctx.enqueue_create_buffer[dtype](SIZE * SIZE)
    out.enqueue_fill(0)
    var a = ctx.enqueue_create_buffer[dtype](SIZE * SIZE)
    a.enqueue_fill(0)
    with a.map_to_host() as a_host:
        for j in range(SIZE):
            for i in range(SIZE):
                k = j * SIZE + i
                a_host[k] = k

    ctx.enqueue_function_checked[add_10, add_10](
            out,
            a,
            UInt(SIZE),
            grid_dim=BLOCKS_PER_GRID,
            block_dim=THREADS_PER_BLOCK,
        )
    
    ctx.synchronize()
    
    with out.map_to_host() as out_host:
        for i in range(SIZE):
            for j in range(SIZE):
                print(out_host[i * SIZE + j])

10.0
11.0
12.0
13.0
14.0
15.0
16.0
17.0
18.0
19.0
20.0
21.0
22.0
23.0
24.0
25.0
26.0
27.0
28.0
29.0
30.0
31.0
32.0
33.0
34.0



## 🎯 Key Takeaways: Handling Thread-Data Mismatches

- **Single block**
    - Get Thread Index, Then Thread Index < Size of vector
- **Multiple Blocks**
    - Get global Thread index, Then Thread Index < Size of vector
- **Mulitple Blocks Multi-Dimension**
    - Get Global Row Index
    - Get Global Column Index
    - Ensure both are less than row and column len of the matrix